In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=4, out_channels=1):
        super().__init__()

        # Downsampling operation
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Decoder: upsampling layers
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)

        # Decoder: convolution blocks
        self.dec4 = DoubleConv(1024, 512)
        self.dec3 = DoubleConv(512, 256)
        self.dec2 = DoubleConv(256, 128)
        self.dec1 = DoubleConv(128, 64)

        # Final segmentation layer
        self.out = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # Return raw pixel scores (logits)
        return self.out(d1)

In [ ]:
import os
import tarfile

DATA_PATH = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
OUTPUT_PATH = "/kaggle/working/brats_data"

os.makedirs(OUTPUT_PATH, exist_ok=True)

for filename in os.listdir(DATA_PATH):

    if not filename.endswith(".tar"):
        continue

    filepath = os.path.join(DATA_PATH, filename)

    print(f"Extracting {filename}...")

    with tarfile.open(filepath, "r") as tar:
        tar.extractall(OUTPUT_PATH)

print("Extraction complete!")

In [ ]:
import os
from sklearn.model_selection import train_test_split

DATA_PATH = "/kaggle/working/brats_data"

modalities = ["flair", "t1", "t1ce", "t2", "seg"]

patients = [
    p for p in os.listdir(DATA_PATH)
    if os.path.isdir(os.path.join(DATA_PATH, p))
]

complete_patients = []

for patient_id in patients:
    patient_path = os.path.join(DATA_PATH, patient_id)

    if all(
        os.path.exists(
            os.path.join(patient_path, f"{patient_id}_{m}.nii.gz")
        )
        for m in modalities
    ):
        complete_patients.append(patient_id)

train_ids, remaining_ids = train_test_split(
    complete_patients,
    test_size=0.30,
    random_state=42
)

val_ids, test_ids = train_test_split(
    remaining_ids,
    test_size=0.50,
    random_state=42
)

print("Training:", len(train_ids))
print("Validation:", len(val_ids))
print("Testing:", len(test_ids))

In [ ]:
from torch.utils.data import Dataset
class BraTSDataset(Dataset):
    def __init__(self, patient_ids, slice_dir):
        ids = set(patient_ids)
        self.files = sorted(
            os.path.join(slice_dir, f) for f in os.listdir(slice_dir)
            if f.rsplit("_", 1)[0] in ids
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        d = np.load(self.files[i])
        X = torch.from_numpy(d["x"]).float()
        Y = torch.from_numpy(d["y"]).unsqueeze(0).float()
        return X, Y

In [ ]:
train_dataset = BraTSDataset(
    patient_ids=train_ids,
    data_path=DATA_PATH
)

val_dataset = BraTSDataset(
    patient_ids=val_ids,
    data_path=DATA_PATH
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Initialize U-Net
model = UNet(in_channels=4, out_channels=1).to(device)

# Binary segmentation loss
criterion = BCEDiceLoss()
# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def normalize(volume):
    mask = volume > 0
    out = np.zeros_like(volume, dtype=np.float32)
    if mask.sum() > 0:
        out[mask] = (volume[mask] - volume[mask].mean()) / (volume[mask].std() + 1e-8)
    return out

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False,
                          num_workers=2, pin_memory=True)

In [ ]:
def dice_score(logits, y):
    p = (torch.sigmoid(logits) > 0.5).float()
    inter = (p * y).sum()
    return ((2 * inter + 1) / (p.sum() + y.sum() + 1)).item()

In [ ]:
X, Y = next(iter(train_loader))

print("Input batch:", X.shape)
print("Target batch:", Y.shape)

In [ ]:
import random
import numpy as np
import nibabel as nib
SLICE_DIR = "/tmp/slices"
os.makedirs(SLICE_DIR, exist_ok=True)

def preprocess(patient_ids, keep_empty_frac=0.2):
    for pid in patient_ids:
        p = os.path.join(DATA_PATH, pid)
        img = np.stack([
            normalize(nib.load(f"{p}/{pid}_{m}.nii.gz").get_fdata().astype(np.float32))
            for m in ["flair", "t1", "t1ce", "t2"]
        ])
        seg = (nib.load(f"{p}/{pid}_seg.nii.gz").get_fdata() > 0).astype(np.float32)
        for s in range(155):
            x, y = img[:, :, :, s], seg[:, :, s]
            if x.max() == x.min():                      
                continue
            if y.sum() == 0 and random.random() > keep_empty_frac:
                continue                                
            np.savez(f"{SLICE_DIR}/{pid}_{s}.npz", x=x, y=y)

preprocess(train_ids)
preprocess(val_ids)

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.4, dice_weight=0.6):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, predictions, targets):
        bce_loss = self.bce(predictions, targets)

        probabilities = torch.sigmoid(predictions)

        intersection = (probabilities * targets).sum()
        dice = (2 * intersection + 1) / (probabilities.sum() + targets.sum() + 1)
        dice_loss = 1 - dice

        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    print("starting training")

    for batch_idx, (X, Y) in enumerate(train_loader):
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()

        predictions = model(X)
        loss = criterion(predictions, Y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if (batch_idx + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}] "
                f"Batch [{batch_idx+1}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f}",
                flush=True
            )

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Average Loss: {running_loss / len(train_loader):.4f}",
        flush=True
    )
    torch.save(
    model.state_dict(),
    f"/kaggle/working/unet_epoch_{epoch+1}.pth"
    )